In [1]:
VALID = {"up", "down", "same", "unknown"}


def describe(value):
    #Turns a qualitative value into a plain-English word.
    return {"up": "increased", "down": "decreased", "same": "stayed the same",
            "unknown": "is unknown"}.get(value, value)


def resolve(a, b, a_name, b_name, outcome, equation, period):
    #Ask for an assumption only when directional arithmetic is ambiguous.
    print(f"\nPeriod {period} \u2014 {outcome}")
    print("-" * 50)
    print(f"{a_name} {describe(a)} while {b_name} {describe(b)} ({equation}).")
    print("The direction cannot be determined from signs alone.")
    while True:
        choice = input("Choose up, down, same, or unknown: ").strip().lower()
        if choice in VALID:
            return choice
        print("Please type: up, down, same, or unknown.")


def explanation(a, b, a_name, b_name, outcome, result, ambiguous):
    #Builds the plain-English sentence describing how the result was found.
    if ambiguous:
        return (f"{a_name} {describe(a)} while {b_name} {describe(b)}. "
                f"{outcome} is ambiguous; user selected {result}.")
    return f"{a_name} {describe(a)} and {b_name} {describe(b)}; {outcome} {describe(result)}."


def directional(a, b, operation, a_name, b_name, outcome, equation, period):
    #Two-term qualitative arithmetic, returning (direction, explanation).
    if a not in VALID or b not in VALID:
        raise ValueError("Directions must be up, down, same, or unknown.")
    if operation in ("add", "multiply"):
        positive = {("up", "up"), ("up", "same"), ("same", "up")}
        negative = {("down", "down"), ("down", "same"), ("same", "down")}
    elif operation in ("subtract", "divide"):
        positive = {("up", "down"), ("up", "same"), ("same", "down")}
        negative = {("down", "up"), ("down", "same"), ("same", "up")}
    else:
        raise ValueError("Unknown operation")

    if (a, b) in positive:
        result, ambiguous = "up", False
    elif (a, b) in negative:
        result, ambiguous = "down", False
    elif (a, b) == ("same", "same"):
        result, ambiguous = "same", False
    else:
        result, ambiguous = resolve(a, b, a_name, b_name, outcome, equation, period), True
    return result, explanation(a, b, a_name, b_name, outcome, result, ambiguous)


def add(a, b, *details):
    return directional(a, b, "add", *details)


def sub(a, b, *details):
    return directional(a, b, "subtract", *details)


def mult(a, b, *details):
    return directional(a, b, "multiply", *details)


def div(a, b, *details):
    return directional(a, b, "divide", *details)


class QualitativePCEX:
    #(display label, internal data key) pairs, in the order shown by print_history().
    table_variables = [
        ("G", "G"), ("a1", "alpha1"), ("a2", "alpha2"), ("theta", "theta"),
        ("r", "r"), ("l20", "lambda20"), ("l21", "lambda21"),
        ("l22", "lambda22"), ("Y", "Y"), ("T", "T"), ("YD", "YD"),
        ("YD^e", "YD_e"), ("C", "C"), ("V", "V"), ("V^e", "V_e"),
        ("B^d", "B_d"), ("H^d", "H_d"), ("B_h", "B_h"), ("B_s", "B_s"),
        ("B_cb", "B_cb"), ("dB_s", "delta_Bs"), ("H_s", "H_s"),
        ("dH_s", "delta_Hs"), ("H_h", "H_h"),
    ]

    def __init__(self, YD_0="same", V_0="same", B_h_0="same"):
        for value in (YD_0, V_0, B_h_0):
            if value not in VALID:
                raise ValueError("Initial values must be qualitative directions.")
        self.period = [0]
        #Every tracked variable starts at "same" by default.
        self.data = {key: ["same"] for _, key in self.table_variables}
        #Stocks and their period-0 lags are seeded from the constructor arguments instead.
        self.data["YD"] = [YD_0]
        self.data["YD_e"] = [YD_0]
        self.data["V"] = [V_0]
        self.data["V_e"] = [V_0]
        self.data["B_h"] = [B_h_0]
        self.data["B_d"] = [B_h_0]
        self.data["B_s"] = [V_0]
        self.data["B_cb"] = ["same"]
        self.data["H_s"] = ["same"]
        self.data["H_h"] = ["same"]
        #One reasoning-trace list per period, built up in calculate_period().
        self.traces = []

    def latest(self, name):
        return self.data[name][-1]

    def calculate_period(self, G, alpha1, alpha2, theta, r, lambda20, lambda21, lambda22):
        inputs = {"G": G, "alpha1": alpha1, "alpha2": alpha2, "theta": theta,
                  "r": r, "lambda20": lambda20, "lambda21": lambda21,
                  "lambda22": lambda22}
        if any(value not in VALID for value in inputs.values()):
            raise ValueError("Every input must be up, down, same, or unknown.")
        p = self.period[-1] + 1
        tr = []

        def calc(name, fn, *args):
            value, text = fn(*args)
            tr.append((name, text))
            return value

        #C = alpha1 * YD^e + alpha2 * V(-1)
        YD_e = self.latest("YD")
        income_effect = calc("Income effect on C", mult, alpha1, YD_e, "alpha1", "expected disposable income", "income effect on consumption", "a1 * YD^e", p)
        wealth_effect = calc("Wealth effect on C", mult, alpha2, self.latest("V"), "alpha2", "previous wealth", "wealth effect on consumption", "a2 * V(-1)", p)
        C = calc("Consumption (C)", add, income_effect, wealth_effect, "income effect", "wealth effect", "consumption (C)", "C = a1*YD^e + a2*V(-1)", p)
        Y = calc("Income (Y)", add, C, G, "consumption", "government spending", "income (Y)", "Y = C + G", p)

        #YD = Y - T + r(-1)*B_h(-1), calculated in two transparent steps.
        household_interest = calc("Household interest income", mult, self.latest("r"), self.latest("B_h"), "previous interest rate", "previous household bills", "household interest income", "r(-1)*B_h(-1)", p)
        tax_base = calc("Tax base", add, Y, household_interest, "income", "household interest income", "tax base", "Y + r(-1)*B_h(-1)", p)
        T = calc("Taxes (T)", mult, theta, tax_base, "tax rate", "tax base", "taxes (T)", "T = theta*(Y + r(-1)*B_h(-1))", p)
        income_after_tax = calc("Income after tax", sub, Y, T, "income", "taxes", "income after tax", "Y - T", p)
        YD = calc("Disposable income (YD)", add, income_after_tax, household_interest, "income after tax", "household interest income", "disposable income (YD)", "YD = Y - T + r(-1)*B_h(-1)", p)

        delta_V = calc("Change in wealth", sub, YD, C, "disposable income", "consumption", "change in wealth", "YD - C", p)
        V = calc("Wealth (V)", add, self.latest("V"), delta_V, "previous wealth", "change in wealth", "wealth (V)", "V = V(-1) + YD - C", p)
        delta_Ve = calc("Expected change in wealth", sub, YD_e, C, "expected disposable income", "consumption", "expected change in wealth", "YD^e - C", p)
        V_e = calc("Expected wealth (V^e)", add, self.latest("V"), delta_Ve, "previous wealth", "expected change in wealth", "expected wealth (V^e)", "V^e = V(-1) + YD^e - C", p)

        #B^d/V^e = lambda20 + lambda21*r(-1) - lambda22*(YD^e/V(-1))
        interest_term = calc("Interest effect on bill share", mult, lambda21, self.latest("r"), "lambda21", "previous interest rate", "interest effect on bill share", "lambda21*r(-1)", p)
        income_wealth_ratio = calc("Expected income/wealth ratio", div, YD_e, self.latest("V"), "expected disposable income", "previous wealth", "expected income/wealth ratio", "YD^e/V(-1)", p)
        ratio_term = calc("Income/wealth effect on bill share", mult, lambda22, income_wealth_ratio, "lambda22", "expected income/wealth ratio", "income/wealth effect on bill share", "lambda22*(YD^e/V(-1))", p)
        share_before_ratio = calc("Bill-share intermediate", add, lambda20, interest_term, "lambda20", "interest effect", "bill-share intermediate", "lambda20 + lambda21*r(-1)", p)
        bill_share = calc("Desired bill share", sub, share_before_ratio, ratio_term, "bill-share intermediate", "income/wealth effect", "desired bill share", "B^d/V^e", p)
        B_d = calc("Desired bills (B^d)", mult, V_e, bill_share, "expected wealth", "desired bill share", "desired bill holdings (B^d)", "B^d = V^e*(B^d/V^e)", p)
        H_d = calc("Desired money (H^d)", sub, V_e, B_d, "expected wealth", "desired bills", "desired money holdings (H^d)", "H^d = V^e - B^d", p)

        #Market clearing and government/central-bank accounting.
        B_h = B_d
        tr.append(("Actual household bills (B_h)", "Asset-market equilibrium: B_h = B^d."))
        government_interest = calc("Government interest payments", mult, self.latest("r"), self.latest("B_s"), "previous interest rate", "previous bills supplied", "government interest payments", "r(-1)*B_s(-1)", p)
        central_bank_interest = calc("Central-bank interest receipts", mult, self.latest("r"), self.latest("B_cb"), "previous interest rate", "previous central-bank bills", "central-bank interest receipts", "r(-1)*B_cb(-1)", p)
        spending = calc("Government outlays", add, G, government_interest, "government spending", "government interest payments", "government outlays", "G + r(-1)*B_s(-1)", p)
        receipts = calc("Government receipts", add, T, central_bank_interest, "taxes", "central-bank interest receipts", "government receipts", "T + r(-1)*B_cb(-1)", p)
        delta_Bs = calc("Change in bills supplied", sub, spending, receipts, "government outlays", "government receipts", "change in bills supplied (dB_s)", "dB_s = outlays - receipts", p)
        B_s = calc("Bills supplied (B_s)", add, self.latest("B_s"), delta_Bs, "previous bills supplied", "change in bills supplied", "bills supplied (B_s)", "B_s = B_s(-1) + dB_s", p)
        B_cb = calc("Central-bank bill holdings", sub, B_s, B_h, "bills supplied", "household bills", "central-bank bill holdings (B_cb)", "B_cb = B_s - B_h", p)
        delta_Hs = calc("Change in money supplied", sub, B_cb, self.latest("B_cb"), "central-bank bill holdings", "previous central-bank bill holdings", "change in money supplied (dH_s)", "dH_s = B_cb - B_cb(-1)", p)
        H_s = calc("Money supplied (H_s)", add, self.latest("H_s"), delta_Hs, "previous money supplied", "change in money supplied", "money supplied (H_s)", "H_s = H_s(-1) + dH_s", p)
        H_h = H_s
        tr.append(("Household money holdings (H_h)", "Redundant accounting identity: H_h = H_s."))

        self.period.append(p)
        values = {**inputs, "Y": Y, "T": T, "YD": YD, "YD_e": YD_e, "C": C,
                  "V": V, "V_e": V_e, "B_d": B_d, "H_d": H_d, "B_h": B_h,
                  "B_s": B_s, "B_cb": B_cb, "delta_Bs": delta_Bs, "H_s": H_s,
                  "delta_Hs": delta_Hs, "H_h": H_h}
        for name, value in values.items():
            self.data[name].append(value)
        self.traces.append(tr)

    def print_history(self):
        #The model has more variables than Chapter 3 but usually only a
        #handful of periods, so variables run down the rows and periods
        #run across the columns, exactly as in the Chapter 3 table.
        label_width, column_width = 10, 10
        header = f"{'Variable':<{label_width}}" + "".join(
            f"{'Period ' + str(p):>{column_width}}" for p in self.period)
        print("=" * len(header))
        print("SIMULATION RESULTS \u2014 Model PCEX1")
        print("=" * len(header))
        print(header)
        print("-" * len(header))
        for label, name in self.table_variables:
            print(f"{label:<{label_width}}" + "".join(
                f"{value:>{column_width}}" for value in self.data[name]))
        print("=" * len(header))

        #Ratio of unknown / same / up / down answers across the table
        #Counts every value in every variable's history, excluding each
        #variable's period-0 initial condition (that is a starting point,
        #not a computed answer). The numbers are unchanged from before
        #only how they're displayed (via _print_answer_distribution) has
        #changed.
        counts = {"up": 0, "down": 0, "same": 0, "unknown": 0}
        for label, name in self.table_variables:
            for value in self.data[name][1:]:
                if value in counts:
                    counts[value] += 1
        total = sum(counts.values())
        if total > 0:
            self._print_answer_distribution(counts, total)

    def _print_answer_distribution(self, counts, total):
        #A small, purely cosmetic "dashboard" box summarising how the
        #up/down/same/unknown answers were distributed across every
        #computed value in the table above. The counts themselves come
        #straight from print_history() - this function only formats them.
        box_width = 64
        inner = box_width - 2

        def boxed(text=""):
            return "\u2502" + text.ljust(inner) + "\u2502"

        top = "\u256d" + "\u2500" * inner + "\u256e"
        divider = "\u251c" + "\u2500" * inner + "\u2524"
        bottom = "\u2570" + "\u2500" * inner + "\u256f"

        print()
        print(top)
        print(boxed(" ANSWER DISTRIBUTION".center(inner)))
        print(divider)
        print(boxed(f"  Unknown : Same : Up : Down  =  {counts['unknown']} : "
                     f"{counts['same']} : {counts['up']} : {counts['down']}"
                     f"   (n = {total})"))
        print(boxed())

        bar_slots = 24
        for key in ("unknown", "same", "up", "down"):
            pct = counts[key] / total
            filled = round(pct * bar_slots)
            bar = "\u2588" * filled + "\u2591" * (bar_slots - filled)
            row = f"  {key.capitalize():<8}{bar}  {pct:>5.1%}"
            print(boxed(row))

        print(bottom)

    def print_explanations(self):
        #Prints the full plain-English reasoning trace for every period.
        #Each variable's heading is underlined and its explanation lines
        #are shown as bullet points - purely a display change, the
        #underlying text is exactly what was generated in calculate_period().
        width = 65
        for period, trace in enumerate(self.traces, start=1):
            print()
            print("=" * width)
            print(f"PERIOD {period} \u2014 REASONING TRACE".center(width))
            print("=" * width)
            for title, text in trace:
                print()
                print(title)
                print("\u2500" * len(title))
                for line_text in text.split("\n"):
                    if line_text.strip():
                        print(f"  \u2022 {line_text}")


econ = QualitativePCEX(YD_0="same", V_0="same", B_h_0="same")
#Period 1: a rise in government spending (G up), everything else same.
econ.calculate_period("up", "same", "same", "same", "same", "same", "same", "same")
#Period 2: no further changes, everything else same.
econ.calculate_period("same", "same", "same", "same", "same", "same", "same", "same")
econ.print_history()
econ.print_explanations()


Period 1 — income after tax
--------------------------------------------------
income increased while taxes increased (Y - T).
The direction cannot be determined from signs alone.
Choose up, down, same, or unknown: same

Period 1 — change in bills supplied (dB_s)
--------------------------------------------------
government outlays increased while government receipts increased (dB_s = outlays - receipts).
The direction cannot be determined from signs alone.
Choose up, down, same, or unknown: up

Period 2 — change in bills supplied (dB_s)
--------------------------------------------------
government outlays increased while government receipts increased (dB_s = outlays - receipts).
The direction cannot be determined from signs alone.
Choose up, down, same, or unknown: down

Period 2 — bills supplied (B_s)
--------------------------------------------------
previous bills supplied increased while change in bills supplied decreased (B_s = B_s(-1) + dB_s).
The direction cannot be determined